- **학습 목표**: "공분산 행렬을 고유값분해하는 것"과 "데이터 행렬을 SVD하는 것"이 수학적으로 동치임을 코드로 직접 검증한다.
- **핵심 개념**: `X = U @ S @ Vᵀ`일 때, `V`의 열이 주성분 방향이고 특이값 `S`와 고유값 `λ` 사이에는 일정한 관계가 있습니다. 그 관계식이 무엇인지는 스스로 찾아내는 것이 오늘의 과제입니다. SVD는 공분산 행렬을 명시적으로 만들지 않으므로 수치적으로 더 안정적입니다.

In [2]:
import numpy as np
import numpy.linalg as linalg
import matplotlib.pyplot as plt

In [35]:
def compare_eig_vs_svd(X_scaled: np.ndarray, k: int = 2):
  """
  요구사항:
  - 방법 A: 공분산 행렬 → np.linalg.eigh → 고유값 내림차순 정렬 → 상위 k개 고유벡터로 투영.
  - 방법 B: np.linalg.svd(X_scaled, full_matrices=False) → U, S, Vt 획득 →
        상위 k개 성분으로 투영.
  - 두 방법의 "고유값"과 "특이값" 사이 변환 공식을 스스로 찾아내서,
  두 값이 일치하는지 np.allclose로 검증한다.
  (힌트: 특이값 S와 샘플 수 n을 사용해 고유값을 만들어낼 수 있다. 공식은 직접 유도할 것.)
  - 투영 결과도 절댓값 기준으로 일치하는지 확인한다 (부호는 다를 수 있음).
  """
  #A-Method
  cov = np.cov(X_scaled, rowvar=False)
  eigenvalue, eigenvector = np.linalg.eigh(cov)
  idx = np.argsort(eigenvalue)[::-1]
  sorted_eigenvalue = eigenvalue[idx]
  sorted_eigenvector = eigenvector[:,idx]
  print(sorted_eigenvalue*(X_scaled.shape[0]-1))
  #B-Method
  U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
  print(S*S)

  #comparison
  print(np.allclose(sorted_eigenvalue*(X_scaled.shape[0]-1), S*S))
X_scaled = np.random.rand(3,3)
#centering
X_scaled -= np.mean(X_scaled, axis=0)
compare_eig_vs_svd(X_scaled,2)

[ 8.42173417e-01  1.97299102e-01 -4.70993911e-18]
[0.84217342 0.1972991  0.        ]
True


- **확인 질문**:
  - 고유값과 특이값 사이 관계식은 무엇이었는가? 왜 그런 관계가 성립하는가 (`XᵀX`를 SVD로 전개해보면 보입니다)?
  - sklearn의 `PCA`는 내부적으로 둘 중 어느 방법을 쓸까? 문서나 소스를 확인해보세요.

고유값과 특이값 사이의 관계식:
A의 고유값= A^TxA의 특이값의 음이 아닌 제곱근
A가 정방행렬일 경우, 고유값=특이값
공분산을 구할때, 샘플n-1로 나누어준다. 이로 인해, A의 고유값에 n-1을 곱한 값과 A의 특이값에 제곱이 같다